# 实践项目 04：XGBoost 脑疾病表格分类

本 Notebook 使用课程提供的 `Data.csv`，每行是一名受试者，`label` 包含 CTL、AD、PD 和 DEP 四类。你将完成数据检查、预处理、XGBoost 训练、四分类评价和变量贡献分析。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码中用整行注释标出了需要填写的位置。先阅读当前单元格的输入、处理和输出，再修改标记区域。合理利用 AI 工具理解问题、学习知识并尝试给出适当的解决方案。

## 任务总览

1. 核对每行的样本单位、类别数量和变量来源。
2. 处理缺失值、类别变量和标识符。
3. 补全 XGBoost 参数并完成训练。
4. 输出混淆矩阵、每类指标和预测概率。
5. 使用置换重要性分析模型依赖的变量。

## 需要保存的结果

`task4_data_summary.png`、`task4_confusion.png`、`task4_roc_pr.png`、`task4_importance.png`、`task4_result.json`。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

SEED = 42; OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
DATA_PATH = None
candidates = sorted(Path('/kaggle/input').rglob('Data.csv'))
if DATA_PATH is None and candidates: DATA_PATH = candidates[0]
assert DATA_PATH is not None, '请挂载包含课程 Data.csv 的数据集。'
df = pd.read_csv(DATA_PATH)
assert {'ID','label'}.issubset(df.columns), '需要课程 Data.csv 中的 ID 和 label 列。'
y_encoder = LabelEncoder(); y = y_encoder.fit_transform(df['label'].astype(str))
X = df.drop(columns=['ID','label'])
print(df.shape, dict(zip(y_encoder.classes_, np.bincount(y))))

## 任务 1：确定样本单位和输入变量

每行对应一名受试者，`ID` 只用于识别，不作为输入。请输出类别数量、变量类型和缺失率。

In [ ]:
# ===== 项目04·任务1·学生填写区（开始） =====
# TODO：输出类别数量、缺失率最高的列、数值列和类别列。
summary = None
# ===== 项目04·任务1·学生填写区（结束） =====
print(summary)

## 任务 2：划分数据、拟合预处理并建立基线

预处理只能使用训练集拟合。验证集和测试集不能参与均值、类别和编码规则的计算。先建立逻辑回归基线，再判断 XGBoost 是否带来额外改进。

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=SEED)
numeric_cols = list(X.select_dtypes(include=np.number).columns)
categorical_cols = [c for c in X.columns if c not in numeric_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])
baseline_pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])
baseline = Pipeline([('pre', baseline_pre), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])
print(len(X_train), len(X_val), len(X_test))

## 任务 3：补全 XGBoost 模型

四分类模型输出四个类别的概率。

In [ ]:
# ===== 项目04·任务3·学生填写区（开始） =====
# TODO：补全 n_estimators、max_depth 和 learning_rate 等轻量参数。
model = Pipeline([('pre', pre), ('clf', XGBClassifier(
    n_estimators=None, max_depth=None, learning_rate=None,
    objective='multi:softprob', num_class=len(y_encoder.classes_),
    subsample=.8, colsample_bytree=.8, eval_metric='mlogloss', random_state=SEED
))])
# ===== 项目04·任务3·学生填写区（结束） =====
print(model)

## 任务 4：评价四分类结果

查看混淆矩阵、每类 precision/recall/F1 和宏平均 F1。

In [ ]:
# ===== 项目04·任务4·学生填写区（开始） =====
# TODO：完成逻辑回归基线与 XGBoost 训练、测试概率、混淆矩阵、宏平均 F1 比较和结果保存。
# ===== 项目04·任务4·学生填写区（结束） =====
print('完成模型后再生成测试结果。')

## 任务 5：变量贡献与错误分析

置换重要性描述当前模型对变量的依赖，不代表变量的生物学因果作用。

In [ ]:
# ===== 项目04·任务5·学生填写区（开始） =====
# TODO：计算置换重要性，查看高置信度错误，并保存 task4_importance.png。
# ===== 项目04·任务5·学生填写区（结束） =====
print('输出文件应写入', OUT)